#Aula 1


In [1]:
!pip install -q --upgrade langchain langchain-google-genai google-generativeai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.1 MB/s eta 0:00:00


Definindo a API KEY do gemini

In [2]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

GOOGLE_API = userdata.get('GEMINI_API')

Definindo o modelo que iremos usar, escolhido foi Gemma pois é open source

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemma-3n-e2b-it",
    temperature=0,
    api_key=GOOGLE_API
)

Teste do llm que definimos

In [4]:
teste = llm.invoke("Quem é você?")
print(teste.content)

Eu sou Gemma, um assistente de IA de código aberto. Sou um modelo de linguagem grande treinado pelo Google DeepMind. Sou um modelo de pesos abertos, amplamente disponível ao público.


Definindo o prompt que irá guiar a mente da nossa LLM, ele é bem definido para a IA não gerar um texto livre, e APENAS JSON.
Separei categorias de decisão na parte Opções e em urgência visando no  fluxo de priorização.

A triagem é justamente a parte de classificaçao da mensagem do User e ai ele sabe para qual nó ele irá ativar (código mais para frente)

In [5]:
triagem_prompt = """
Analise a pergunta do usuário sobre o projeto Prisma e classifique em UMA destas categorias:

**Opções:**
- "conceito_metodologico": Dúvidas sobre pesquisa empírica/explícita, metodologia
- "garantia_etica": Preocupações com anonimato, privacidade, ética
- "performance_resposta": Perguntas sobre errar, mentir, resposta certa/errada
- "natureza_teste": Questionamentos se é teste psicológico, diagnóstico
- "duvida_geral": Outras questões sobre o projeto Prisma
- "fora_do_escopo": Não relacionado ao projeto Prisma

**Classifique também a urgência:**
- "alta": Preocupações éticas, ansiedade do participante
- "media": Dúvidas conceituais importantes
- "baixa": Curiosidades gerais

Pergunta: {pergunta_usuario}

Responda APENAS com JSON:
{{
    "decisao": "categoria",
    "urgencia": "alta|media|baixa",
    "campos_faltantes": []
}}
"""

Importamos o BaseModel para que ele verifique se o JSON que está retornando da IA está correto!
essa classe atua como o molde que a IA vai retornar!

In [6]:
from pydantic import BaseModel
from typing import Literal, List

class TriagemOut(BaseModel):
    decisao: Literal[
        "conceito_metodologico",
        "garantia_etica",
        "performance_resposta",
        "natureza_teste",
        "duvida_geral",
        "fora_do_escopo"
    ]
    urgencia: Literal["alta", "media", "baixa"]
    campos_faltantes: List[str]

Mesma coisa da definição de cima.

In [7]:
llm_triagem = ChatGoogleGenerativeAI(
    model="gemma-3n-e2b-it",
    temperature=0,
    api_key=GOOGLE_API
)

As importaçoes são para criar o formato da mensagem, tanto da IA (System) e do Usuário (human).
a função triagem ela analisa a pergunta e mostra para qual direcionamento a pergunta está levando ela

In [8]:
from langchain_core.messages import SystemMessage,  HumanMessage

                #quando rodar o LLM, vai ser de forma estruturada igual definimos no TriagemOut
triagem_chain = llm_triagem.with_structured_output(TriagemOut)

def triagem(mensagem: str):
    """Versão simplificada para Gemma """
    print(f"Analisando pergunta: {mensagem}")

    # Lógica simples baseada em palavras-chave (igual ao tutorial)
    mensagem_lower = mensagem.lower()

    #detecta as palavras
    if "errar" in mensagem_lower or "mentir" in mensagem_lower or "certo" in mensagem_lower:
        return {"decisao": "performance_resposta", "urgencia": "media", "campos_faltantes": []}
    elif "anonim" in mensagem_lower or "privacid" in mensagem_lower or "dado" in mensagem_lower:
        return {"decisao": "garantia_etica", "urgencia": "alta", "campos_faltantes": []}
    elif "empíric" in mensagem_lower or "explícit" in mensagem_lower or "metodolog" in mensagem_lower:
        return {"decisao": "conceito_metodologico", "urgencia": "media", "campos_faltantes": []}
    elif "psicológico" in mensagem_lower or "teste" in mensagem_lower or "diagnóstico" in mensagem_lower:
        return {"decisao": "natureza_teste", "urgencia": "media", "campos_faltantes": []}
    else:
        return {"decisao": "duvida_geral", "urgencia": "baixa", "campos_faltantes": []}


In [9]:
testes = ["Para quer serve esses testes?", "O que acontece se eu errar?"]

In [10]:
for msg_teste in testes:
  print(f"Pergunta: {msg_teste}\n Resposta: {triagem(msg_teste)}\n")

Analisando pergunta: Para quer serve esses testes?
Pergunta: Para quer serve esses testes?
 Resposta: {'decisao': 'natureza_teste', 'urgencia': 'media', 'campos_faltantes': []}

Analisando pergunta: O que acontece se eu errar?
Pergunta: O que acontece se eu errar?
 Resposta: {'decisao': 'performance_resposta', 'urgencia': 'media', 'campos_faltantes': []}



#Aula 2


Colocando os pdfs para a IA aprender com o PyMuPDFLoader

In [11]:
!pip install -q --upgrade langchain_community faiss-cpu langchain-text-splitters pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [12]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

docs = []

for n in Path("/content/").glob("*.pdf"):
    try:
      loader = PyMuPDFLoader(str(n))
      docs.extend(loader.load())
      print(f"Carregado arquivo {n.name}")
    except Exception as e:
      print(f"Erro carregar o arquivo {n.name}")

print(f"Docs carregados = {len(docs)}")

Carregado arquivo Explicacao_Teórica_PRISMA.pdf
Carregado arquivo Documentação Prisma - 1.0.pdf
Carregado arquivo Explicacao_Geral_PRISMA.pdf
Docs carregados = 11


Aqui é para separar os PDFs em splitters, você passa dois números que vai ser a quantidade de palavras que vai pegar por chunks, e o quanto vai se repetir (overlap) para continuar com a lógica que, caso não tiver isso, a informação pode ficar desestruturada.

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(docs)

O conteúdo do pdf já está dentro da variável chunks. Agora próxima parte é ver a proximidade entre as palavras: embeddings.
Os embeddings são a forma de transformar texto em vetores numéricos, e as palavras semelhantes ficarão num vetor próximo.

Esse model que se usa em baixo é próprio para o embeddings e open-source

In [14]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"  # ou multilingual-e5-base
)


/tmp/ipython-input-4223415286.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS = É uma biblioteca que faz buscas rápidas em vetores.
Ele pega os chunks e

Cria um vector store, cada chunks que ele recebe passa pelo HuggingFaceEmbeddings model que a gente definiu a cima.

Retriever pega a pergunta do user, transforma as palavras em vetores com embedding também e compara o vetor das perguntas com os das respostas! A gente define um limite de similaridade que a gente aceita passar. Nesse caso, maior que 0.3, e retorna os 4 chunks mais relevantes

In [15]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)

# vai pegar a pergunta do user e comparar com os chunks vetoricamente por similaridade

retriever = vectorstore.as_retriever(search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.3, "k": 4})

# maior limite, mais restrintivo é

RAG = Retrieval-Augmented Generation
ou seja, Geração Aumentada por Recuperação.

Junta o Retriever (PDF, BD...) e Generation (LLM)
Então é graças a ele que a nossa IA aprende com nossos PDFs e responde com LLM

Os chunks gerados dos códigos passados depois de carregar os DOCs, dividir ele com splitters, transformar em vetor e pegar os semelhantes e deixarem próximos, pegar a pergunta do usuário, passar pelo embedding e o retriever comparar entre o PDF e pergunta os próximos (embedding) os chunks que passarem por essa ultima etapa vão para o RAG

Esse é outro prompt para guiar a IA, mas essa é voltada a guiar a LLM a procurar a informaçao no doc

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

prompt_rag = ChatPromptTemplate.from_messages(
    [
        ("""
        Você é assistente da empresa Prisma que ajuda somente com base no
        contexto fornecido, se não souber responder com base nos PDFs diga
        'Não sei' e explique o porque não sabe.
        """
        ),
        ("human", "Pergunta: {input}:\nContexto:\n{context}")
    ]
)


# cria cadeia que passa lista de documentos

document_chain = create_stuff_documents_chain(llm_triagem, prompt_rag)

Na função perguntar_RAG pegamos os docs que passaram na avaliaçao do retriever
e auxilia como a IA responde dependendo se tem informaçao ou não

In [17]:
from typing import Dict

def perguntar_RAG(pergunta: str) -> Dict:
  docs_relacionados = retriever.invoke(pergunta)
  if not docs_relacionados:
    return {"answer": "Não sei....",
            "citacoes": [],
            "contexto_encontrado": False}

  answer = document_chain.invoke({"input": pergunta,
                                "context": docs_relacionados})

  txt = (answer or "").strip()

  if txt.rstrip(".!?") == "Não sei":
    return {"answer":"Não sei",
            "citacoes": [],
            "contexto_encontrado": False}

  return {"answer": txt,
          "citacoes": docs_relacionados,
          "contexto_encontrado": True}

Primeiro teste com perguntas direcionadas aos PDFs. Pegamos a funçao passada e mandamos as perguntas que criamos para ela achar a resposta no PDF.
Aproveitamos e botamos as citaçoes para sabermos onde ele encontrou tal informaçao.

In [18]:
for msg_teste in testes:
  resposta = perguntar_RAG(msg_teste)
  print(f"Pergunta: {msg_teste}\n Resposta: {resposta['answer']}\n")
  if resposta['contexto_encontrado']:
    print(f"CITAÇOES: {resposta['citacoes']}")
    print("===============================")

Pergunta: Para quer serve esses testes?
 Resposta: Não sei....

Pergunta: O que acontece se eu errar?
 Resposta: Não sei....



#Aula 3

Fazer os nós para montar o grafo

In [19]:
!pip install -q --upgrade langgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 15.4 MB/s eta 0:00:00


Criamos o AgentState para ser o bem comum entre todos os nós. Tem todas as informaçoes da pergunta do User.
Ele e passado como argumento para todos os nós pois ele está com as informaçoes que deve sempre aparecer na saida, indepedente do nó atinginte.

In [20]:
from typing import TypedDict, Optional
class AgentState(TypedDict, total = False):
  mensagem: str
  triagem: Dict
  resposta: Optional[str]
  citacoes: List[dict]
  rag_sucesso: bool
  acao_final: str


Esse nó triagem tem a  mesma funçao do promp_triagem = classificar.
ele só recebe a mensagem e passa pela triagem

In [21]:
def node_triagem(state: AgentState) -> AgentState:
  print("Executando nó triagem!!!")
  return {"triagem": triagem(state["mensagem"])}

É o no do RAG, ela pega a mensagem e nela faz a embeddings e a retriever.

In [22]:
def node_auto_resolver(state: AgentState) -> AgentState:
  print("Executando nó RAG!!!")
  resposta_rag = perguntar_RAG(state["mensagem"])
  update: AgentState = {
      "resposta": resposta_rag["answer"],
      "citacoes": resposta_rag.get("citacoes", []),
      "rag_sucesso": resposta_rag["contexto_encontrado"]
  }
  if resposta_rag["contexto_encontrado"]:
    update["acao_final"] = "AUTO_RESOLVER"

  return update

As palavras que cairam o "campos_faltantes" da triagem vão para a triagem e juntam todas elas num vetor.


In [23]:
def node_pedindo_info(state: AgentState) -> AgentState:
  print("Executando nó pedindo infor!!!")
  faltantes = state["triagem"].get("campos_faltantes", [])
  detalhe = ",".join(faltantes) if faltantes else "Tema e contexto especifico"

  if faltantes:
    detalhe = ",".join(faltantes)
  else:
    detalhe = "Tema e contexto especifico"

  return {"resposta": f"Para avançar, preciso de detalhe: {detalhe}",
          "citacoes": [],
          "acao_final": "PEDIR_INFO"
          }

## Arestas

decidir_principal serve para decidir para qual caminho a mensagem do user mais se enquadra. Dependendo da pergunta, essa funçao faz a ligaçao da triagem para o nó que cair

In [24]:
def decidir_principal(state: AgentState) -> str:
    print("Decidindo após a triagem")
    decisao = state["triagem"]["decisao"]

    if decisao in ["conceito_metodologico", "natureza_teste", "duvida_geral"]:
        return "auto_resolver"
    elif decisao == "fora_do_escopo":
        return "pedir_info"
    elif decisao == "garantia_etica":
        return "finalizar"
    else:
        return "finalizar"



Ele so é acesso se cair no auto_resolver, que é basicamente feedback. Ele é necessário pois como é ligado a pesquisa, pode ou não achaar nos chunks.

In [25]:
def decidir_pos_auto_resolver(state: AgentState) -> str:
    print("Decidindo após o auto_resolver...")

    if state.get("rag_sucesso"):
        print("RAG com sucesso, finalizando o fluxo.")
        return "ok"

    # Se o RAG falhar → não tem abrir_chamado, então só pede mais detalhe
    print("RAG falhou, vou pedir mais informações...")
    return "pedir_info"


In [26]:
from langgraph.graph import StateGraph, START, END

# o grafo! ele vai passar o AgentState em todos os nós
workflow = StateGraph(AgentState)

# Nós principais
# (nome, e o a funçao que vai rodar)
workflow.add_node("auto_resolver", node_auto_resolver)
workflow.add_node("pedir_info", node_pedindo_info)
workflow.add_node("triagem", node_triagem)

# Define o nó finalizar igual aos outros
def node_finalizar(state: AgentState) -> AgentState:
    print("Executando nó finalizar!!!")
    return {
        "resposta": state.get("resposta", "Não sei responder."),
        "citacoes": state.get("citacoes", []),
        "acao_final": "finalizar"
    }


workflow.add_node("finalizar", node_finalizar)

# Arestas
# Sempre começa com o "triagem"
workflow.add_edge(START, "triagem")

# ligando a triagem, chama a funçao decidir_principal para ver o final da ligação
workflow.add_conditional_edges(
    "triagem", decidir_principal,
    {
        "auto_resolver": "auto_resolver",
        "pedir_info": "pedir_info",
        "finalizar": "finalizar",
    }
)


# ligando o auto_resolver com o resultado da func pos_auto_resolver
workflow.add_conditional_edges(
    "auto_resolver", decidir_pos_auto_resolver,
    {
        "ok": END,
        "pedir_info": "pedir_info",
    }
)

# caminhando para o final do grafo
workflow.add_edge("pedir_info", "finalizar")
workflow.add_edge("finalizar", END)

# Compilar grafo
grafo = workflow.compile()


Descrever o esboço do grafo feito. 👉 Site: https://mermaid.live

In [27]:
!apt-get update -qq
!apt-get -qq install graphviz graphviz-dev libgraphviz-dev pkg-config
!pip install pygraphviz

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 126441 files and directories currently installed.)
Removing r-base-dev (4.5.1-1.2204.0) ...
dpkg: pkgconf: dependency problems, but removing anyway as you requested:
 libsndfile1-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libopencv-dev depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libmkl-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libjack-dev depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libgphoto2-dev:amd64 

In [28]:
from IPython.display import display, Image

print(grafo.get_graph().draw_mermaid())


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	auto_resolver(auto_resolver)
	pedir_info(pedir_info)
	triagem(triagem)
	finalizar(finalizar)
	__end__([<p>__end__</p>]):::last
	__start__ --> triagem;
	auto_resolver -. &nbsp;ok&nbsp; .-> __end__;
	auto_resolver -.-> pedir_info;
	pedir_info --> finalizar;
	triagem -.-> auto_resolver;
	triagem -.-> finalizar;
	triagem -.-> pedir_info;
	finalizar --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



Testar mais perguntas

In [29]:
perguntas = [
    "Quem pode responder esse teste?",
    "Minhas respostas vão ser anônimas?",

    "Quem desenvolveu o Prisma?",
    "Qual o preço da gasolina hoje?"
]

In [30]:
for msg_test in perguntas:
    resposta_final = grafo.invoke({"mensagem": msg_test})
    triag = resposta_final.get("triagem", {})

    print(f"PERGUNTA: {msg_test}")
    print(f"DECISÃO: {triag.get('decisao')} | "
          f"URGÊNCIA: {triag.get('urgencia')} | "
          f"AÇÃO FINAL: {resposta_final.get('acao_final')}")
    print(f"RESPOSTA: {resposta_final.get('resposta')}")

    if resposta_final.get("citacoes"):
        print("CITAÇÕES:")
        for citacao in resposta_final.get("citacoes", []):
            print(f"  - Documento: {citacao.metadata.get('source')}, "
                  f"Página: {citacao.metadata.get('page')}")
            print(f"    Trecho: {citacao.page_content}")
    print("-----------------------------------------")


Executando nó triagem!!!
Analisando pergunta: Quem pode responder esse teste?
Decidindo após a triagem
Executando nó RAG!!!
Decidindo após o auto_resolver...
RAG com sucesso, finalizando o fluxo.
PERGUNTA: Quem pode responder esse teste?
DECISÃO: natureza_teste | URGÊNCIA: media | AÇÃO FINAL: AUTO_RESOLVER
RESPOSTA: Com base no contexto fornecido, o pesquisador(a) pode responder ao teste. O texto indica que o pesquisador(a) pode definir critérios para identificar e destacar respostas inválidas, sugerindo que ele(a) é quem avalia e interpreta os resultados.
CITAÇÕES:
  - Documento: /content/Documentação Prisma - 1.0.pdf, Página: 3
    Trecho: de dados
inválidos
O pesquisador(a)
pode definir
critérios como
latência mínima
(ex: < 300ms) ou
percentual de
erro (ex: > 10%)
para que o
sistema
identifique e
destaque essas
respostas.
Implementado
O
participanteService.ts
possui a lógica para
sinalizar sessões
inválidas com base
em critérios de
  - Documento: /content/Explicacao_Teórica_PRISMA.p

Decidindo após o auto_resolver...
RAG com sucesso, finalizando o fluxo.
PERGUNTA: Quem desenvolveu o Prisma?
DECISÃO: duvida_geral | URGÊNCIA: baixa | AÇÃO FINAL: AUTO_RESOLVER
RESPOSTA: De acordo com o contexto fornecido, o Prisma foi desenvolvido no Vortex (Laboratório de Inovação em Computação da Universidade de Fortaleza – Unifor) por Amanda Fonseca Rodrigues, Kaylany e Luma, alunas de Ciência da Computação.

Portanto, a resposta é: **Amanda Fonseca Rodrigues, Kaylany e Luma**.
CITAÇÕES:
  - Documento: /content/Explicacao_Teórica_PRISMA.pdf, Página: 0
    Trecho: Projeto PRISMA – Explicação Geral
1. Origem e Autoria
O PRISMA é um projeto desenvolvido no Vortex (Laboratório de Inovação em
Computação da Universidade de Fortaleza – Unifor). Ele foi criado por Amanda Fonseca
Rodrigues, Kaylany e Luma, alunas de Ciência da Computação.
2. Objetivo
  - Documento: /content/Explicacao_Geral_PRISMA.pdf, Página: 0
    Trecho: Projeto PRISMA – Explicação Geral
1. Origem e Autoria
O PRISMA é um

Mesma coisa, mas menos informaçoes

In [32]:
for msg_test in perguntas:
    resposta_final = grafo.invoke({"mensagem": msg_test})
    triag = resposta_final.get("triagem", {})

    print("══════════════════════════════════════════")
    print(f"❓ PERGUNTA: {msg_test}")
    print()
    print(f"💬 RESPOSTA:\n{resposta_final.get('resposta')}")
    print("══════════════════════════════════════════\n")


Executando nó triagem!!!
Analisando pergunta: Quem pode responder esse teste?
Decidindo após a triagem
Executando nó RAG!!!
Decidindo após o auto_resolver...
RAG com sucesso, finalizando o fluxo.
══════════════════════════════════════════
❓ PERGUNTA: Quem pode responder esse teste?

💬 RESPOSTA:
Com base no contexto fornecido, o pesquisador(a) pode responder ao teste. O texto indica que o pesquisador(a) pode definir critérios para identificar e destacar respostas inválidas, sugerindo que ele(a) é quem avalia e interpreta os resultados.
══════════════════════════════════════════

Executando nó triagem!!!
Analisando pergunta: Minhas respostas vão ser anônimas?
Decidindo após a triagem
Executando nó RAG!!!
Decidindo após o auto_resolver...
RAG com sucesso, finalizando o fluxo.
══════════════════════════════════════════
❓ PERGUNTA: Minhas respostas vão ser anônimas?

💬 RESPOSTA:
Sim, suas respostas serão anônimas. De acordo com o contexto fornecido, "Todos os testes são anônimos" e "Nenhuma

Decidindo após o auto_resolver...
RAG com sucesso, finalizando o fluxo.
══════════════════════════════════════════
❓ PERGUNTA: Quem desenvolveu o Prisma?

💬 RESPOSTA:
De acordo com o contexto fornecido, o Prisma foi desenvolvido no Vortex (Laboratório de Inovação em Computação da Universidade de Fortaleza – Unifor) por Amanda Fonseca Rodrigues, Kaylany e Luma, alunas de Ciência da Computação.

Portanto, a resposta é: **Amanda Fonseca Rodrigues, Kaylany e Luma**.
══════════════════════════════════════════

Executando nó triagem!!!
Analisando pergunta: Qual o preço da gasolina hoje?
Decidindo após a triagem
Executando nó RAG!!!
Decidindo após o auto_resolver...
RAG falhou, vou pedir mais informações...
Executando nó pedindo infor!!!
Executando nó finalizar!!!
══════════════════════════════════════════
❓ PERGUNTA: Qual o preço da gasolina hoje?

💬 RESPOSTA:
Para avançar, preciso de detalhe: Tema e contexto especifico
══════════════════════════════════════════

